<!-- 학습 보강 셀 -->

# 03. Node 학습 흐름

이 노트북은 `Document`와 `Node`의 차이를 이해하는 데 초점을 둡니다.
Document가 원본 문서라면, Node는 검색과 임베딩에 쓰기 위해 잘라낸 조각입니다.

### 1. Node 객체 직접 생성

<!-- 학습 보강 셀 -->

## 왜 Document를 바로 검색하지 않고 Node로 나눌까?

긴 문서 전체를 하나의 벡터로 만들면 질문과 관련된 작은 부분을 찾기 어렵습니다.
Node로 나누면 검색 단위가 작아져서 특정 문장이나 문단을 더 정확히 찾을 수 있습니다.
RAG 품질은 Node를 얼마나 적절한 크기로 나누는지에 크게 영향을 받습니다.

In [1]:
from llama_index.core import Document
from llama_index.core.schema import TextNode

# Document는 원본 문서이고, Node는 검색/임베딩에 사용하기 위해 나눈 작은 조각입니다.
document = Document(
    text="""인공지능은 우리의 미래를 변화시킬 것입니다.
이러한 변화에 우리는 준비되어 있어야 합니다.""",
    id_='ai_future_doc',
)

# 문서를 2개의 노드로 직접 분할합니다.
# - TextNode의 고유 ID는 id_로 지정합니다.
# - 원본 Document와의 연결 정보는 metadata에 남겨 추적할 수 있게 합니다.
# - 기존처럼 고정 글자 수로 자르면 문장이 중간에서 끊길 수 있어 문장 단위로 나눕니다.
sentences = [line.strip() for line in document.text.splitlines() if line.strip()]
node1 = TextNode(
    text=sentences[0],
    id_='ai_future_node_1',
    metadata={'source_document_id': document.id_},
)
node2 = TextNode(
    text=sentences[1],
    id_='ai_future_node_2',
    metadata={'source_document_id': document.id_},
)

print(node1)
print(node2)

Node ID: ai_future_node_1
Text: 인공지능은 우리의 미래를 변화시킬 것입니다.
Node ID: ai_future_node_2
Text: 이러한 변화에 우리는 준비되어 있어야 합니다.


### 2. 문서를 불러와서 Document와 Node 객체 생성


In [2]:
# Word(.docx) 파일을 읽으려면 docx2txt가 필요합니다.
# !pip install docx2txt

<!-- 학습 보강 셀 -->

## 파일 형식별 추가 패키지

PDF, Word, Excel, 웹 페이지처럼 파일 형식이 달라지면 내부 파서도 달라집니다.
`SimpleDirectoryReader`는 공통 인터페이스를 제공하지만, 실제 파일을 읽기 위해서는 형식별 의존성이 필요할 수 있습니다.
실행 오류가 나면 먼저 해당 파일 형식의 reader 패키지가 설치되어 있는지 확인하세요.

In [3]:
from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import SentenceSplitter

In [4]:
# NewData 폴더에서 docx 파일만 읽습니다.
# required_exts를 지정하면 다른 확장자는 무시되므로 예제가 안정적으로 실행됩니다.
documents = SimpleDirectoryReader(
    input_dir='../NewData',
    required_exts=['.docx'],
).load_data()
print('읽어온 문서 수:', len(documents))

읽어온 문서 수: 1


In [5]:
# 읽어온 Word 문서의 본문 일부를 확인합니다.
print(documents[0].text[:1000])

제목: "Node의 이해: AI의 정보 조각 만들기"



1. Node란 무엇일까요?

Node는 큰 문서를 작은 조각으로 나눈 것입니다. 마치 긴 책을 여러 장의 카드로 나누어 정리하는 것과 비슷합니다. 예를 들어, 역사책 한 권을 시대별로 나누어 카드를 만드는 것처럼, 긴 문서를 여러 개의 Node로 나눕니다.



2. Node는 왜 필요할까요?

큰 문서를 그대로 사용하면 AI가 정보를 찾고 이해하는 데 어려움이 있습니다. 마치 학생이 시험 공부를 할 때 두꺼운 교과서를 한 번에 읽는 것보다, 단원별로 나누어 공부하는 것이 더 효과적인 것처럼, AI도 작은 조각으로 나눈 정보를 더 잘 활용할 수 있습니다.



3. Node의 특징은 무엇일까요?

- 적절한 크기: 한 Node는 AI가 한 번에 처리하기 좋은 크기로 만듭니다.

- 문맥 유지: 내용이 잘리더라도 의미가 통하도록 적절히 나눕니다.

- 관계 유지: 원본 문서와의 연결 정보를 보관합니다.

- 메타데이터: 각 Node가 어디서 왔는지, 어떤 내용인지 알 수 있는 정보를 담습니다.



4. Node는 어떻게 만들어질까요?

Node를 만드는 방법은 여러 가지가 있습니다:

- 문장 단위로 나누기: 문장을 기준으로 나눕니다.

- 단락 단위로 나누기: 내용이 바뀌는 단락을 기준으로 나눕니다.

- 길이 기준으로 나누기: 일정한 길이를 기준으로 나눕니다.



5. Node의 활용

Node는 다음과 같은 상황에서 유용하게 사용됩니다:

- 정보 검색: 필요한 정보를 빠르게 찾을 수 있습니다.

- 질문 답변: 관련된 Node만 활용하여 정확한 답변을 만듭니다.

- 정보 요약: 여러 Node의 내용을 모아 요약할 수 있습니다.



6. Node 사용의 장점

- 효율적인 검색: 작은 단위로 나누어져 있어 검색이 빠릅니다.

- 정확한 답변: 필요한 부분만 정확하게 활용할 수 있습니다.

- 메모리 효율: 필요한 Node만 메모리에 불러올 수 있습니다.


In [6]:
# SentenceSplitter는 문장 경계를 최대한 보존하면서 문서를 Node로 분할합니다.
# chunk_size는 한 Node의 최대 토큰 수, chunk_overlap은 앞뒤 Node 사이에 겹쳐 둘 토큰 수입니다.
parser = SentenceSplitter(
    chunk_size=200,
    chunk_overlap=20,
)

nodes = parser.get_nodes_from_documents(documents)
print('생성된 Node 수:', len(nodes))

생성된 Node 수: 6


<!-- 학습 보강 셀 -->

## chunk_size와 chunk_overlap 감각 잡기

`chunk_size`가 너무 작으면 문맥이 잘려 답변 품질이 떨어질 수 있고, 너무 크면 검색이 둔해질 수 있습니다.
`chunk_overlap`은 앞뒤 조각 사이에 문맥을 조금 겹쳐 두는 장치입니다.
일반적으로 문단 단위 의미가 유지되는지 출력된 Node를 직접 읽어 보며 조정합니다.

In [7]:
# Node 확인
# - enumerate(..., start=1)을 사용하면 출력 번호가 사람이 읽기 편한 1부터 시작합니다.
for i, node in enumerate(nodes, start=1):
    print(f'=== Node {i} ===')
    print(node.text)
    print('-' * 80)

=== Node 1 ===
제목: "Node의 이해: AI의 정보 조각 만들기"



1. Node란 무엇일까요?

Node는 큰 문서를 작은 조각으로 나눈 것입니다. 마치 긴 책을 여러 장의 카드로 나누어 정리하는 것과 비슷합니다. 예를 들어, 역사책 한 권을 시대별로 나누어 카드를 만드는 것처럼, 긴 문서를 여러 개의 Node로 나눕니다.
--------------------------------------------------------------------------------
=== Node 2 ===
2. Node는 왜 필요할까요?

큰 문서를 그대로 사용하면 AI가 정보를 찾고 이해하는 데 어려움이 있습니다. 마치 학생이 시험 공부를 할 때 두꺼운 교과서를 한 번에 읽는 것보다, 단원별로 나누어 공부하는 것이 더 효과적인 것처럼, AI도 작은 조각으로 나눈 정보를 더 잘 활용할 수 있습니다.
--------------------------------------------------------------------------------
=== Node 3 ===
3. Node의 특징은 무엇일까요?

- 적절한 크기: 한 Node는 AI가 한 번에 처리하기 좋은 크기로 만듭니다.

- 문맥 유지: 내용이 잘리더라도 의미가 통하도록 적절히 나눕니다.

- 관계 유지: 원본 문서와의 연결 정보를 보관합니다.

- 메타데이터: 각 Node가 어디서 왔는지, 어떤 내용인지 알 수 있는 정보를 담습니다.
--------------------------------------------------------------------------------
=== Node 4 ===
4. Node는 어떻게 만들어질까요?

Node를 만드는 방법은 여러 가지가 있습니다:

- 문장 단위로 나누기: 문장을 기준으로 나눕니다.

- 단락 단위로 나누기: 내용이 바뀌는 단락을 기준으로 나눕니다.

- 길이 기준으로 나누기: 일정한 길이를 기준으로 나눕니다.
-----------

In [8]:
# 첫 번째 Node의 메타데이터 확인
# - 이전 셀의 반복문 변수 node에 의존하지 않도록 nodes[0]을 직접 사용합니다.
nodes[0].metadata

{'file_name': 'word_example.docx',
 'file_path': '/Users/cheng80/Documents/WorkSpace/RAG/NewNote/../NewData/word_example.docx',
 'file_type': 'application/vnd.openxmlformats-officedocument.wordprocessingml.document',
 'file_size': 15420,
 'creation_date': '2025-12-20',
 'last_modified_date': '2025-12-20'}